# Feature Engineering – Daily Net Revenue

This notebook loads the supplied daily revenue and transaction datasets, checks their consistency, creates a complete daily series, and builds leakage-safe calendar, lag and rolling-window features.

**Leakage rule:** every feature for date *t* is calculated from observations strictly before *t*. Rolling statistics are calculated from the historical series ending at *t-1*.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

DATA_DIR = Path(".")  # Put both CSV files in this folder, or change this path.
DAILY_FILE = DATA_DIR / "daily_net_revenue.csv"
TRANSACTIONS_FILE = DATA_DIR / "sales_transactions.csv"

daily = pd.read_csv(DAILY_FILE, parse_dates=["Date"])
tx = pd.read_csv(TRANSACTIONS_FILE, parse_dates=["Date", "InvoiceDate"])
daily.head(), daily.shape, tx.shape

## 1. Validate and prepare the daily target

The daily file is the forecasting target. Missing calendar dates are filled with zero because the series represents daily sales revenue; duplicate dates are summed.

In [ ]:
daily["Date"] = pd.to_datetime(daily["Date"]).dt.normalize()
daily["NetRevenue"] = pd.to_numeric(daily["NetRevenue"], errors="coerce")
daily = daily.dropna(subset=["Date", "NetRevenue"])
daily = daily.groupby("Date", as_index=False)["NetRevenue"].sum()

calendar = pd.date_range(daily.Date.min(), daily.Date.max(), freq="D")
y = (daily.set_index("Date")["NetRevenue"].sort_index()
     .reindex(calendar, fill_value=0.0))
y.index.name = "Date"
y.name = "NetRevenue"

print("Date range:", y.index.min(), "to", y.index.max())
print("Calendar days:", len(y), "| Missing dates filled:", len(calendar) - daily.Date.nunique())
y.describe()

## 2. Reconcile against transaction-level revenue

The supplied transaction file contains a `Revenue` field. Summing it by `Date` should reproduce `daily_net_revenue.csv` for dates represented in the transaction file.

In [ ]:
tx["Date"] = pd.to_datetime(tx["Date"]).dt.normalize()
tx["Revenue"] = pd.to_numeric(tx["Revenue"], errors="coerce")
tx_daily = tx.dropna(subset=["Date", "Revenue"]).groupby("Date")["Revenue"].sum()

reconciliation = y.rename("DailyFileRevenue").to_frame().join(
    tx_daily.rename("TransactionRevenue"), how="outer"
).fillna(0)
reconciliation["Difference"] = (
    reconciliation["DailyFileRevenue"] - reconciliation["TransactionRevenue"]
)
print("Maximum absolute difference:",
      reconciliation["Difference"].abs().max())
reconciliation.head()

## 3. Create leakage-safe features

Calendar features are known in advance. Lag and rolling features use only past target values. The first 28 observations are omitted from the supervised training matrix because the model requires a 28-day lag.

In [ ]:
LAGS = (1, 2, 3, 7, 14, 21, 28)
ROLLING_WINDOWS = (7, 14, 28)

def make_features(history: pd.Series, forecast_date: pd.Timestamp) -> pd.DataFrame:
    history = history.astype(float)
    row = {
        "dayofweek": forecast_date.dayofweek,
        "day": forecast_date.day,
        "month": forecast_date.month,
        "quarter": forecast_date.quarter,
        "year": forecast_date.year,
        "dayofyear": forecast_date.dayofyear,
        "weekofyear": int(forecast_date.isocalendar().week),
        "is_weekend": int(forecast_date.dayofweek >= 5),
        "is_month_start": int(forecast_date.is_month_start),
        "is_month_end": int(forecast_date.is_month_end),
    }
    for lag in LAGS:
        row[f"lag_{lag}"] = float(history.iloc[-lag])
    for window in ROLLING_WINDOWS:
        past = history.iloc[-window:]
        row[f"rolling_mean_{window}"] = float(past.mean())
        row[f"rolling_std_{window}"] = float(past.std(ddof=0))
        row[f"rolling_median_{window}"] = float(past.median())
    return pd.DataFrame([row], index=[forecast_date])

feature_rows = []
for i in range(max(LAGS), len(y)):
    feature_rows.append(make_features(y.iloc[:i], y.index[i]))
X = pd.concat(feature_rows)
target = y.loc[X.index].rename("target")

features = X.join(target)
print("Feature matrix:", X.shape)
features.head()

## 4. Export engineered features

The exported table is useful for inspection and model development. The target is kept in a separate column; it is not included among the input features.

In [ ]:
features.to_csv("engineered_daily_features.csv", index_label="Date")
print("Saved engineered_daily_features.csv")